# Chapter 1 — Data Loading, Cleaning and Preparation

**Project:** Policyholder Risk Stratification and Claim Probability Prediction

**Dataset:** Porto Seguro Safe Driver Prediction (Kaggle)

**Author:** Whitney Kemuma

---

### Chapter Overview

This chapter covers the full data preparation pipeline:

| Step | Activity |
|------|----------|
| 1.1 | Load raw data from the data folder |
| 1.2 | Initial inspection — shape, types, missing values |
| 1.3 | Data cleaning — handle missing values, remove duplicates |
| 1.4 | Feature engineering — encode categoricals, create risk flag |
| 1.5 | Train-test split — stratified 80/20 split |
| 1.6 | Feature scaling — StandardScaler for Logistic Regression |
| 1.7 | Save cleaned data to the data folder |

> **Note on Class Imbalance:** This dataset is heavily imbalanced — only ~3.6% of
> policyholders filed a claim. Rather than applying SMOTE oversampling (which is 
> computationally expensive on 595,000 records), this study uses
> **class_weight='balanced'** in all models. This approach adjusts the loss function
> to penalise misclassification of the minority class more heavily, achieving
> imbalance correction without generating synthetic data.


# ── 1.1 IMPORTS ──────────────────────────────────────────────
### 1.1 Import Libraries

All libraries used in this chapter are imported below with brief annotations
explaining their role in the pipeline.

In [1]:
# ── Standard libraries ───────────────────────────────────────
import numpy as np                        # Numerical operations
import pandas as pd                       # Data manipulation
import os                                 # File path handling
import warnings
warnings.filterwarnings('ignore')

# ── Preprocessing ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split   # Train-test split
from sklearn.preprocessing import StandardScaler       # Feature scaling

print("Libraries imported successfully.")


Libraries imported successfully.


# ── 1.2 LOAD DATA ────────────────────────────────────────────
### 1.2 Load Raw Data

The raw dataset is loaded directly from the `data/` folder. The path is constructed
relative to this notebook's location so the project runs on any machine without
hardcoded paths.


In [2]:
DATA_PATH = os.path.join(
    os.path.dirname(os.getcwd()),  # project root
    'data',
    'train.csv.zip'                # zip file name
)

df = pd.read_csv(DATA_PATH, compression='zip')

print("Loaded from  :", DATA_PATH)
print("Dataset shape:", df.shape)
print("\\nFirst 5 rows:")
df.head()


Loaded from  : C:\Users\Administrator\Desktop\policyholder-risk-predictions\data\train.csv.zip
Dataset shape: (595212, 59)
\nFirst 5 rows:


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


In [3]:
# Data types and memory usage
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595212 entries, 0 to 595211
Data columns (total 59 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              595212 non-null  int64  
 1   target          595212 non-null  int64  
 2   ps_ind_01       595212 non-null  int64  
 3   ps_ind_02_cat   595212 non-null  int64  
 4   ps_ind_03       595212 non-null  int64  
 5   ps_ind_04_cat   595212 non-null  int64  
 6   ps_ind_05_cat   595212 non-null  int64  
 7   ps_ind_06_bin   595212 non-null  int64  
 8   ps_ind_07_bin   595212 non-null  int64  
 9   ps_ind_08_bin   595212 non-null  int64  
 10  ps_ind_09_bin   595212 non-null  int64  
 11  ps_ind_10_bin   595212 non-null  int64  
 12  ps_ind_11_bin   595212 non-null  int64  
 13  ps_ind_12_bin   595212 non-null  int64  
 14  ps_ind_13_bin   595212 non-null  int64  
 15  ps_ind_14       595212 non-null  int64  
 16  ps_ind_15       595212 non-null  int64  
 17  ps_ind_16_

In [4]:
# Statistical summary — check ranges, means, and any anomalies
df.describe()


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,5.952120e+05,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,...,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000
mean,7.438036e+05,0.036448,1.900378,1.358943,4.423318,0.416794,0.405188,0.393742,0.257033,0.163921,...,5.441382,1.441918,2.872288,7.539026,0.122427,0.627840,0.554182,0.287182,0.349024,0.153318
std,4.293678e+05,0.187401,1.983789,0.664594,2.699902,0.493311,1.350642,0.488579,0.436998,0.370205,...,2.332871,1.202963,1.694887,2.746652,0.327779,0.483381,0.497056,0.452447,0.476662,0.360295
min,7.000000e+00,0.000000,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.719915e+05,0.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.435475e+05,0.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.000000,1.000000,3.000000,7.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,1.115549e+06,0.000000,3.000000,2.000000,6.000000,1.000000,0.000000,1.000000,1.000000,0.000000,...,7.000000,2.000000,4.000000,9.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.488027e+06,1.000000,7.000000,4.000000,11.000000,1.000000,6.000000,1.000000,1.000000,1.000000,...,19.000000,10.000000,13.000000,23.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# ── 1.3 INITIAL INSPECTION ───────────────────────────────────
### 1.3 Initial Inspection

Before cleaning, we examine the data for three things:
- **Missing values** — Porto Seguro encodes missing as `-1`
- **Class distribution** — confirms the imbalance we need to handle
- **Duplicate rows** — checks data integrity

In [5]:
# ── Target class distribution ────────────────────────────────

print(" Target Variable Distribution:")
print(df['target'].value_counts())
print("\\nClass proportions (%):")
print((df['target'].value_counts(normalize=True) * 100).round(2))
print("\\n  Note: Dataset is heavily imbalanced.")
print("   Strategy: class_weight='balanced' applied at model training stage.")

 Target Variable Distribution:
target
0    573518
1     21694
Name: count, dtype: int64
\nClass proportions (%):
target
0    96.36
1     3.64
Name: proportion, dtype: float64
\n  Note: Dataset is heavily imbalanced.
   Strategy: class_weight='balanced' applied at model training stage.


In [6]:
# ── Check for -1 encoded missing values ──────────────────────
# Porto Seguro uses -1 to represent missing — we count them per column
missing_mask = (df == -1)
missing_count = missing_mask.sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count (-1)': missing_count,
    'Missing %'         : missing_pct
}).query('`Missing Count (-1)` > 0').sort_values('Missing %', ascending=False)

print(f" Columns with -1 encoded missing values: {len(missing_summary)}")
missing_summary


 Columns with -1 encoded missing values: 13


,Missing Count (-1),Missing %
ps_car_03_cat,411231,69.09
ps_car_05_cat,266551,44.78
ps_reg_03,107772,18.11
ps_car_14,42620,7.16
ps_car_07_cat,11489,1.93
ps_ind_05_cat,5809,0.98
ps_car_09_cat,569,0.10
ps_ind_02_cat,216,0.04
ps_car_01_cat,107,0.02
ps_ind_04_cat,83,0.01


In [7]:
# ── Check for duplicate rows ─────────────────────────────────
duplicates = df.duplicated().sum()
print(f" Duplicate rows found: {duplicates}")


 Duplicate rows found: 0


# ── 1.4 DATA CLEANING ────────────────────────────────────────
### 1.4 Data Cleaning

Three cleaning steps are applied:

1. **Drop the ID column** — `id` is a row identifier with no predictive value
2. **Replace -1 with NaN** — converts Porto Seguro's missing value encoding
   to standard Python NaN for proper handling
3. **Impute missing values** — numeric columns filled with median (robust to
   outliers), categorical columns filled with mode (most frequent value)

> **Methodological note:** Median imputation is preferred over mean imputation
> for insurance data because claim-related features often contain outliers that
> would skew a mean-based fill.

In [8]:
# ── Step 1: Drop ID column ───────────────────────────────────
df.drop(columns=['id'], inplace=True)
print(" Dropped 'id' column.")

# ── Step 2: Replace -1 with NaN ──────────────────────────────
df.replace(-1, np.nan, inplace=True)
print(" Replaced -1 encoded missing values with NaN.")

# ── Step 3: Impute missing values ────────────────────────────
# Numeric  → median  (robust to outliers)
# Object   → mode    (most frequent category)
imputed_cols = []
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
        imputed_cols.append(col)

print(f" Imputed missing values in {len(imputed_cols)} columns.")
print("   Columns imputed:", imputed_cols)


 Dropped 'id' column.
 Replaced -1 encoded missing values with NaN.
 Imputed missing values in 13 columns.
   Columns imputed: ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_reg_03', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_05_cat', 'ps_car_07_cat', 'ps_car_09_cat', 'ps_car_11', 'ps_car_12', 'ps_car_14']


In [9]:
# ── Step 4: Remove duplicates ────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
after  = len(df)
print(f" Removed {before - after} duplicate rows.")
print(f" Clean dataset shape: {df.shape}")

 Removed 0 duplicate rows.
 Clean dataset shape: (595212, 58)


In [10]:
# ── Verify: no missing values remain ─────────────────────────
remaining_nulls = df.isnull().sum().sum()
print(f" Remaining missing values: {remaining_nulls}")
assert remaining_nulls == 0, " Missing values still present — review imputation step"
print(" Data is clean. No missing values remain.")

 Remaining missing values: 0
 Data is clean. No missing values remain.


# ── 1.5 FEATURE ENGINEERING ──────────────────────────────────
### 1.5 Feature Engineering

Two feature engineering steps are applied:

**a) One-hot encoding of categorical features**
Columns ending in `_cat` are categorical variables. These are encoded using
one-hot encoding with `drop_first=True` to avoid the dummy variable trap
(multicollinearity between encoded columns).

**b) Risk flag count feature**
A new feature `risk_flag_sum` is created by summing all binary (`_bin`) columns
for each policyholder. This captures the total number of risk indicators present —
a higher count suggests a higher-risk profile. This type of aggregated risk feature
is commonly used in actuarial scoring models.


In [11]:
# ── a) One-hot encode categorical columns ────────────────────
cat_cols = [col for col in df.columns if col.endswith('_cat')]
print(f" Categorical columns found: {len(cat_cols)}")
print("   Columns:", cat_cols)

# drop_first=True removes one dummy per group to avoid multicollinearity
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"\\n Encoding complete. New shape: {df.shape}")

 Categorical columns found: 14
   Columns: ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat', 'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat']
\n Encoding complete. New shape: (595212, 205)


In [12]:
# ── b) Create risk flag sum feature ─────────────────────────
# Sum of all binary risk indicator columns per policyholder
# Higher value = more risk indicators activated
bin_cols = [col for col in df.columns if col.endswith('_bin')]
df['risk_flag_sum'] = df[bin_cols].sum(axis=1)

print(f" Created 'risk_flag_sum' from {len(bin_cols)} binary columns.")
print(f"   Feature range: {df['risk_flag_sum'].min()} – {df['risk_flag_sum'].max()}")
print(f"   Mean risk flags per policyholder: {df['risk_flag_sum'].mean():.2f}")


 Created 'risk_flag_sum' from 17 binary columns.
   Feature range: 1 – 10
   Mean risk flags per policyholder: 4.04


# ── 1.6 TRAIN-TEST SPLIT ─────────────────────────────────────
### 1.6 Train-Test Split

The dataset is split into 80% training and 20% test sets using a **stratified**
split. Stratification ensures that the class ratio (3.6% claims) is preserved
in both sets — this is essential for meaningful evaluation on imbalanced data.

The test set is held out entirely and used only for final model evaluation in
Chapter 4. It is never used during model training or hyperparameter tuning.

In [13]:
# ── Define features and target ───────────────────────────────
X = df.drop(columns=['target'])   # Feature matrix
y = df['target']                  # Target: 1 = claim, 0 = no claim

# ── Stratified 80/20 split ────────────────────────────────────
# stratify=y preserves the 3.6% claim rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f" Training set : {X_train.shape[0]:,} rows | {X_train.shape[1]} features")
print(f" Test set     : {X_test.shape[0]:,} rows  | {X_test.shape[1]} features")
print(f"\\nTraining class distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
print(f"\\nTest class distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')


 Training set : 476,169 rows | 205 features
 Test set     : 119,043 rows  | 205 features
\nTraining class distribution:
target
0    96.36%
1     3.64%
Name: proportion, dtype: object
\nTest class distribution:
target
0    96.36%
1     3.64%
Name: proportion, dtype: object


# ── 1.7 FEATURE SCALING ──────────────────────────────────────
### 1.7 Feature Scaling

StandardScaler is applied to normalise features to mean = 0, standard deviation = 1.

**Why scaling is needed:** Logistic Regression uses a gradient-based optimisation
algorithm that is sensitive to the scale of input features. Without scaling, features
with large ranges dominate the gradient updates and the model converges slowly or
produces biased coefficients.

**Important:** The scaler is **fit only on the training data** and then applied to
the test data. Fitting on the full dataset before splitting would constitute data
leakage — the model would have indirect knowledge of the test set during training.

> **Note:** Random Forest and XGBoost do not require scaling as they are tree-based
> models that split on thresholds, not distances.


In [14]:
# ── Fit scaler on training data only ─────────────────────────
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit + transform train
X_test_scaled  = scaler.transform(X_test)         # Transform test only

print(" Feature scaling complete.")
print(f"   Training set mean (first feature): {X_train_scaled[:, 0].mean():.6f}  → should be ~0")
print(f"   Training set std  (first feature): {X_train_scaled[:, 0].std():.6f}   → should be ~1")

 Feature scaling complete.
   Training set mean (first feature): -0.000000  → should be ~0
   Training set std  (first feature): 1.000000   → should be ~1


# ── 1.8 SAVE CLEANED DATA ────────────────────────────────────
### 1.8 Save Cleaned Data

The cleaned and engineered dataset is saved to the `data/` folder as
`train_cleaned.csv`. This file is used as the starting point in Chapters 2–5,
ensuring all chapters work from the same cleaned dataset without repeating
the cleaning pipeline unnecessarily.

The train/test split arrays are also saved as CSV files for reproducibility.

In [15]:
# ── Build path to data folder ────────────────────────────────
DATA_OUT = os.path.join(os.path.dirname(os.getcwd()), 'data')

# ── Save full cleaned dataset ─────────────────────────────────
cleaned_path = os.path.join(DATA_OUT, 'train_cleaned.csv')
df.to_csv(cleaned_path, index=False)
print(f" Cleaned dataset saved  → {cleaned_path}")
print(f"   Shape: {df.shape}")

# ── Save train and test splits ────────────────────────────────
X_train_path = os.path.join(DATA_OUT, 'X_train.csv')
X_test_path  = os.path.join(DATA_OUT, 'X_test.csv')
y_train_path = os.path.join(DATA_OUT, 'y_train.csv')
y_test_path  = os.path.join(DATA_OUT, 'y_test.csv')

X_train.to_csv(X_train_path, index=False)
X_test.to_csv(X_test_path,   index=False)
y_train.to_csv(y_train_path, index=False)
y_test.to_csv(y_test_path,   index=False)

print(f" X_train saved → {X_train_path}")
print(f" X_test  saved → {X_test_path}")
print(f" y_train saved → {y_train_path}")
print(f" y_test  saved → {y_test_path}")

 Cleaned dataset saved  → C:\Users\Administrator\Desktop\policyholder-risk-predictions\data\train_cleaned.csv
   Shape: (595212, 206)
 X_train saved → C:\Users\Administrator\Desktop\policyholder-risk-predictions\data\X_train.csv
 X_test  saved → C:\Users\Administrator\Desktop\policyholder-risk-predictions\data\X_test.csv
 y_train saved → C:\Users\Administrator\Desktop\policyholder-risk-predictions\data\y_train.csv
 y_test  saved → C:\Users\Administrator\Desktop\policyholder-risk-predictions\data\y_test.csv


In [16]:
# ── Final chapter summary ─────────────────────────────────────
print("=" * 55)
print("  CHAPTER 1 COMPLETE — DATA PREPARATION SUMMARY")
print("=" * 55)
print(f"  Raw dataset rows         : 595,212")
print(f"  Cleaned dataset rows     : {len(df):,}")
print(f"  Total features           : {X.shape[1]}")
print(f"  Training samples         : {len(X_train):,}")
print(f"  Test samples             : {len(X_test):,}")
print(f"  Class imbalance strategy : class_weight='balanced'")
print(f"  Files saved to data/     : 5 files")
print("=" * 55)
print("\\n  Proceed to Chapter 2 — Exploratory Data Analysis")


  CHAPTER 1 COMPLETE — DATA PREPARATION SUMMARY
  Raw dataset rows         : 595,212
  Cleaned dataset rows     : 595,212
  Total features           : 205
  Training samples         : 476,169
  Test samples             : 119,043
  Class imbalance strategy : class_weight='balanced'
  Files saved to data/     : 5 files
\n  Proceed to Chapter 2 — Exploratory Data Analysis
